# cgSENSE Loop
In this notebook  a cgSENSE reconstruction can be executed. It is possible to iterate on more datasets with the for loop. For that it is just needed to insert the right names of the datasets in the for loop.

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

import sys
import os

sys.path.insert(0, "../src")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import zarr

from juart.conopt.functional.fourier import nonuniform_fourier_transform_adjoint
from juart.conopt.tfs.fourier import nonuniform_transfer_function
from juart.recon.sense import SENSE
from juart.vis.interactive import InteractiveFigure3D

device='cuda:1'
saving = True # whether the single images should be saved or not. The save file will have the same name as the dataset
cgiter = 125 # number of cg iterations

for string in ["fibo_phantom_128spk_R1",
               'fibo_phantom_128spk_R2_6765points',
               'fibo_phantom_128spk_R4_4181points',
               'fibo_phantom_128spk_R8_1597points',
               'fibo_phantom_128spk_R16_987points',
               'fibo_phantom_128spk_R32_610points',
               'fibo_phantom_128spk_R64_233points',
               'fibo_phantom_128spk_R128_144points']:

    directory = string
    store = zarr.storage.LocalStore(
        f"/home/jovyan/datasets/{directory}/",
    )
    group = zarr.open_group(store, mode="r")
    k = group["k"][:]
    C = group["C"][:]
    d = group["d"][:]
    
    # Scale trajectory to [-0.5, 0.5]
    k = k / (2 * k.max())
    
    k = torch.from_numpy(k)
    C = torch.from_numpy(C)
    d = torch.from_numpy(d)
    
    kspace_mask = torch.randint(0,2,(1,k.shape[1]))
    k_masked = k*kspace_mask
    kspace_mask.shape, k_masked.shape, d.shape, C.shape
    
    AHd = nonuniform_fourier_transform_adjoint(k_masked[...,None,None], d[...,None,None], (128, 128, 128))
    AHd = torch.sum(torch.conj(C[...,None,None]) * AHd, dim=0, keepdim=True)
    
    H = nonuniform_transfer_function(k*kspace_mask, (1, 128, 128, 128), oversampling=(2, 2, 2))
    
    C = C[..., None]
    H = H[..., None, None]

    print(k.shape,C.shape,d.shape,AHd.shape,H.shape)
    
    cg_solver = SENSE(
        C.to(device),
        AHd.to(device),
        H.to(device),
        axes=(1, 2, 3),
        maxiter=cgiter,
        verbose=True,
        device=device
    )
    
    shape = (128, 128, 128)
    cg_image = cg_solver.solve().view(torch.complex64).reshape(shape)
    
    if saving:
        if not os.path.isdir(f"/home/jovyan/images/num_cgsense_reco/{directory}"):
            os.makedirs(f"/home/jovyan/images/num_cgsense_reco/{directory}")
        
        torch.save(cg_image, f'/home/jovyan/images/num_cgsense_reco/{directory}/{cgiter}i')
        print("image saved successfully")